![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 01 | Framing and Baseline

*Project 3 — AI Product: From Model to Product*

**Student:** Gabriel Gomes

**Track:** LLM

**Data:** Criterion Collection

---
## Setup

In [ ]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/
sys.path.append("../src")      # the src scripts import each other by bare name (`from first_look import ...`)

import pandas as pd

%reload_ext autoreload
%autoreload 2
from src.first_look import *
from src.data_quality import *
from src.explore_texts import *
from src.prepare_documents import *
from src.eval_checks import *
from src.functions import *

pd.set_option("display.max_colwidth", 200)

---
## 1. Framing the product

- **The user:** A film fan or cineclub organizer who is picking what to watch next from the Criterion Collection and wants a recommendation they can trust, not a generic “best films” list. A typical situation is planning a screening night: “What should I watch if I want a tense, 1970s political thriller?”
- **The input and the output:** They type a natural-language request (a mood, decade, country, director, or “something like X”). They get back about three films from the Criterion catalog, each with a one-line reason and citations to its Wikipedia article and TMDB data (year, genres, runtime, poster). If the question can’t be answered from the sources, such as live streaming availability, it says so instead of guessing.
- **The decision:** They choose which film or films to watch or program, for example a double feature. They can follow the citations to read more before committing, so the reasons need to be checkable.
- **The metric:** I’ll score precision@3 on 30 pre-written questions. The automatic checks confirm that every pick is in the catalog and meets the hard constraints (decade, genre, country, runtime), and I’ll compare against an LLM-alone baseline. The costliest mistake is a wrong answer, such as recommending a film that isn’t in the Collection or stating a false fact, because the user may act on it. An answer with no source is next, since the user can’t verify it. An unnecessary refusal costs least, as long as the system still refuses all five truly unanswerable questions. I’ll track the hit rate of the cited sources, the false-refusal rate on answerable questions, and the refusal rate on the five unanswerable ones.

---
## 2. Loading the data

Run `python download_data.py` from the repo root first. The files land in `data/raw/` and `data/processed/`.

In [2]:
films, manifest = load_tables()

---
## 3. First look

**What one row is.** The table has 1,928 rows and 24 columns, and one row is one distinct film in the Criterion Collection. Each row joins the Criterion list on Wikipedia (title, year, director, spine number, box sets) with TMDB (genres, countries, runtime, keywords, overview, poster) and, through `wiki_title`, with one Wikipedia article.

**How complete the join is.** All 1,928 films matched TMDB, although 57 of those matches are low-confidence. 1,763 films link to a downloaded article, and the other 165 can only be answered from their list and TMDB fields.

**No column is a clean key.** Five rows share a `wiki_title` (Che and The Human Condition, whose parts are separate films), and four more reach a shared article through a redirect (the two parts of Ivan the Terrible and of Fårö Document). The only five rows that share a TMDB ID are the Small Axe films, which TMDB lists as one series, and 475 rows share a spine number, because a film with no spine of its own takes its box set's. Spine numbers are also missing for 273 films (14.2%), so they cannot identify a film on their own.

**Gaps and shape.** Most fields are nearly complete. The gap that matters most is `keywords`, missing for 10.5% of films, which limits the keyword cross-check on the mood questions. Missing `tagline` (42.6%) and `sets` (56.5%) values are expected, since taglines are optional and most films were not released in a box set. The catalog leans toward mid-century cinema, with 61% of films from the 1950s to the 1980s, and TMDB overviews are short (a median of 39 words), so most of the depth the system can draw on is in the Wikipedia articles.

In [3]:
films.head()

,title,year,director,film_spine,in_main_table,laserdisc_only,wiki_title,sets,revid,tmdb_id,...,languages,original_language,tagline,overview,keywords,tmdb_directors,top_cast,poster_url,tmdb_url,match_confidence
0,Grand Illusion,1937,Jean Renoir,1.0,True,False,La Grande Illusion,Essential Art House: 50 Years of Janus Films,1383674996,777,...,"[Russian, German, French, English]",fr,A Great Drama of Human Emotions,"A group of French soldiers, including the patrician Captain de Boeldieu and the working-class Lieutenant Maréchal, grapple with their own class differences after being captured and held in a World...","[prisoner, france, countryside, escape, germany, world war i, prisoner of war, brotherhood, camp, aristocrat, fraternity, war injury, escaped prisoner, prison camp, trying to escape, escape plan, ...",[Jean Renoir],"[Jean Gabin, Dita Parlo, Pierre Fresnay, Erich von Stroheim, Julien Carette, Georges Péclet, Werner Florian, Jean Dasté]",https://image.tmdb.org/t/p/w780/lWg41zE0FVixkNsFgxnlRyvDYv9.jpg,https://www.themoviedb.org/movie/777,high
1,Seven Samurai,1954,Akira Kurosawa,2.0,True,False,Seven Samurai,AK 100: 25 Films by Akira Kurosawa | Essential Art House: 50 Years of Janus Films,1383674996,346,...,[Japanese],ja,Unmatched for suspense and spectacle!,"A samurai answers a village's request for protection after he falls on hard times. The town needs protection from bandits, so the samurai gathers six others to help him teach the people how to def...","[martial arts, japan, samurai, sword, peasant, village, looting, rice, fencing, moral ambiguity, forbidden love, black and white, battle, period drama, bandit, practice, farmer, jidaigeki, 16th ce...",[Akira Kurosawa],"[Toshirō Mifune, Takashi Shimura, Yoshio Inaba, Seiji Miyaguchi, Minoru Chiaki, Daisuke Katō, Isao Kimura, Keiko Tsushima]",https://image.tmdb.org/t/p/w780/lOMGc8bnSwQhS4XyE1S99uH8NXf.jpg,https://www.themoviedb.org/movie/346,high
2,King Kong,1933,Merian Cooper,2.0,True,True,King Kong (1933 film),NaN,1383674996,244,...,[English],en,Beauty conquered the beast!,"Adventurous filmmaker Carl Denham sets out to produce a motion picture unlike anything the world has seen before. Alongside his leading lady Ann Darrow and his first mate Jack Driscoll, they arriv...","[new york city, ship, exotic island, island, monster, unsociability, screenplay, movie business, great depression, dinosaur, black and white, pre-code, sea voyage, damsel in distress, empire state...","[Merian C. Cooper, Ernest B. Schoedsack]","[Robert Armstrong, Fay Wray, Bruce Cabot, Frank Reicher, Victor Wong, James Flavin, Sam Hardy, Noble Johnson]",https://image.tmdb.org/t/p/w780/lHlnxKL5GbgRibyRFI7n1Ey850i.jpg,https://www.themoviedb.org/movie/244,high
3,The Lady Vanishes,1938,Alfred Hitchcock,3.0,True,False,The Lady Vanishes,Essential Art House: 50 Years of Janus Films,1383674996,940,...,"[Italian, English, German, French]",en,Comedy! Chills! Chuckles! in a Mystery Express!,"On a train headed for England a group of travelers is delayed by an avalanche. Holed up in a hotel in a fictional European country, young Iris befriends elderly Miss Froy. When the train resumes, ...","[espionage, secret agent, flowerpot, concussion, search for witnesses, conspiracy, black and white, train, british spy, old lady, missing person, magician, bold]",[Alfred Hitchcock],"[Margaret Lockwood, Michael Redgrave, Paul Lukas, May Whitty, Basil Radford, Naunton Wayne, Cecil Parker, Linden Travers]",https://image.tmdb.org/t/p/w780/c1t9LB76LvEARPanfEzXmkm7fwY.jpg,https://www.themoviedb.org/movie/940,high
4,Amarcord,1973,Federico Fellini,4.0,True,False,Amarcord,Essential Fellini,1383674996,7857,...,"[Greek, Italian]",it,The Fantastic World of Fellini!,"In an Italian seaside town, young Titta gets into trouble with his friends and watches various local eccentrics as they engage in often absurd behavior. Frequently clashing with his stern father a...","[small town, fascism, surrealism, sheik, coming of age, rural area, bathtub, se

In [4]:
manifest.head()

,requested_title,fetched_at,status,title,pageid,revid,wikibase_item,bytes,year_in_page,director_in_page
0,La Grande Illusion,2026-11-07,ok,La Grande Illusion,62097,1379930960,Q470336,170435.0,True,True
1,Seven Samurai,2026-11-07,ok,Seven Samurai,31371,1381655382,Q189540,483546.0,True,True
2,King Kong (1933 film),2026-11-07,ok,King Kong (1933 film),16782,1384090500,Q309048,416021.0,True,True
3,The Lady Vanishes,2026-11-07,ok,The Lady Vanishes,9404578,1378628565,Q500044,159700.0,True,True
4,Amarcord,2026-11-07,ok,Amarcord,239086,1377742482,Q18428,193704.0,True,True


In [5]:
overview(films, "criterion_films_enriched").sort_values(by='missing_%', ascending=False)

criterion_films_enriched: 1,928 rows x 24 columns


,dtype,non_null,missing_%,n_unique,example
sets,str,838,56.5,167,Essential Art House: 50 Years of Janus …
tagline,str,1107,42.6,1103,A Great Drama of Human Emotions
film_spine,float64,1655,14.2,1306,1.0
keywords,object,1726,10.5,1675,"['prisoner', 'france', 'countryside', '…"
wiki_title,str,1763,8.6,1760,La Grande Illusion
top_cast,object,1813,6.0,1806,"['Jean Gabin', 'Dita Parlo', 'Pierre Fr…"
genres,object,1891,1.9,417,"['Drama', 'History', 'War']"
countries,object,1913,0.8,232,['France']
languages,object,1914,0.7,341,"['Russian', 'German', 'French', 'Englis…"
director,str,1918,0.5,711,Jean Renoir


In [6]:
read_in_full(films, n=5, seed=0)   # 5 rows, nothing truncated

row 706
  title:                A Room with a View
  year:                 1986
  director:             James Ivory
  film_spine:           775.0
  in_main_table:        True
  laserdisc_only:       False
  wiki_title:           A Room with a View (1985 film)
  sets:                 (missing)
  revid:                1383674996
  tmdb_id:              11257
  tmdb_original_title:  A Room with a View
  runtime:              117.0
  genres:               ['Drama', 'Romance']
  countries:            ['United Kingdom']
  languages:            ['English', 'Italian']
  original_language:    en
  tagline:              An Edwardian Romance
  overview:             When Lucy Honeychurch and chaperon Charlotte Bartlett find themselves in
                        Florence with rooms without views, fellow guests Mr Emerson and son George
                        step in to remedy the situation. Meeting the Emersons could change Lucy's
                        life forever but, once back in England, how

In [7]:
check_key(films, ["wiki_title"])                             # what is one row?

['wiki_title']: 1,763 rows with a value, NOT unique (5 rows share a key); 165 missing


,title,year,director,film_spine,in_main_table,laserdisc_only,wiki_title,sets,revid,tmdb_id,...,languages,original_language,tagline,overview,keywords,tmdb_directors,top_cast,poster_url,tmdb_url,match_confidence
489,Che: Part One,2008,Steven Soderbergh,496.0,True,False,Che (2008 film),NaN,1383674996,8881,...,"[English, Spanish]",en,Everyone knows the icon. Few know the man.,"Ernesto Guevara, known as 'Che', leads a group of Cuban exiles under Fidel Castro in a revolution to overthrow Fulgencio Batista, the dictator of Cuba.","[hero, central intelligence agency (cia), cuba, biography, che guevara, fidel castro, cuban revolution, 1950s]",[Steven Soderbergh],"[Benicio del Toro, Demián Bichir, Santiago Cabrera, Vladimir Cruz, Alfredo de Quesada, Jsu Garcia, Kahlil Méndez, Elvira Mínguez]",https://image.tmdb.org/t/p/w780/jWgPiVTmJsHWKVjAy2gvSFUH9yA.jpg,https://www.themoviedb.org/movie/8881,high
490,Che: Part Two,2008,Steven Soderbergh,496.0,True,False,Che (2008 film),NaN,1383674996,8880,...,"[English, Spanish]",en,The untold story of his final revolution,"Seven years after his triumph in Cuba, Che winds up in Bolivia, where he tries to ignite the same revolutionary fires as before.","[hero, central intelligence agency (cia), cuba, biography, che guevara, bolivian army, la paz, bolivia]",[Steven Soderbergh],"[Benicio del Toro, Carlos Bardem, Demián Bichir, Joaquim de Almeida, Pablo Durán, Eduard Fernández, Marc-André Grondin, Óscar Jaenada]",https://image.tmdb.org/t/p/w780/fKQwwWXYQNkrZa59VbnfVy2kfoF.jpg,https://www.themoviedb.org/movie/8880,high
472,The Human Condition I: No Greater Love,1959,Masaki Kobayashi,480.0,True,False,The Human Condition (film series),NaN,1383674996,31217,...,"[Mandarin, Japanese]",ja,The Immortal Story.,"After handing in a report on the treatment of Chinese colonial labor, Kaji is offered the post of labour chief at a large mining operation in Manchuria, which also grants him exemption from milita...","[japan, fascism, world war ii, prison escape, conscientious objector, snow, soldier, chinese, shallow grave, mandarin, ore mine, philosophical, complex, depressed, dramatic, depressing, romantic, ...",[Masaki Kobayashi],"[Tatsuya Nakadai, Michiyo Aratama, Chikage Awashima, Ineko Arima, So Yamamura, Akira Ishihama, Kōji Nanbara, Seiji Miyaguchi]",https://image.tmdb.org/t/p/w780/q7godavx5XEbeVxEeqi4pmiOKdI.jpg,https://www.themoviedb.org/movie/31217,high
473,The Human Condition II: Road to Eternity,1959,Masaki Kobayashi,480.0,True,False,The Human Condition (film series),NaN,1383674996,34528,...,"[Russian, Mandarin, Japanese]",ja,The second one,"Kaji, sent to the Japanese army labeled Red, witnesses cruelties in the army and revolts against the abusive treatment against a fellow recruit. He also sees his friend Shinjô defecting to the Rus...","[japan, world war ii, recruit, sequel, conscientious objector, pacific war, military, japanese army, foxhole, trenches, imperial japan, philosophical, complex, depressed, dramatic, depressing, rom...",[Masaki Kobayashi],"[Tatsuya Nakadai, Michiyo Aratama, Kokinji Katsura, Jun Tatara, Michirō Minami, Kei Satō, Kunie Tanaka, Ryōhei Uchida]",https://image.tmdb.org/t/p/w780/7jfv1C4t06VGenQxf3HbmLVmUHZ.jpg,https://www.themoviedb.org/movie/34528,high
474,The Human Condition III: A Soldier's Prayer,1961,Masaki Kobayashi,480.0,True,False,The Human Condition (film series),NaN,1383674996,34530,...,"[Mandarin, Japanese, Russian]",ja,Love is the Condition for Being Human.,"After the Japanese defeat to the Russians, Kaji leads the last remaining men through Manchuria. Intent on returning to his old life, he faces great odds in a variety of different harrowing circums...","[japan, china, world war ii, prisoner of war, manchuria, sequel, soldier, chinese, japanese army, philosophical, complex, depressed, dramatic, depressing, romantic, complicated, informative, tragic]",[Masaki Kobayashi],"[Tatsuya Nakadai, Michiyo Aratama, Tamao Nakamura, Yūsuke Kawazu, Chishū Ryū, Taketoshi Naitō, Kyôko 

In [8]:
joined = coverage(films, manifest)                           # films -> Wikipedia article -> TMDB

films: 1,928
  with a Wikipedia article link: 1,763
  article downloaded (status ok): 1,763
  matched to TMDB:                1,928
  article AND TMDB (usable):      1,763
article_status
ok                 1763
no article link     165


In [9]:
print(article_text(joined.loc[joined["article_status"] == "ok", "wiki_title"].iloc[0]))

[infobox] Directed by: Jean Renoir
[infobox] Written by: Charles Spaak Jean Renoir
[infobox] Produced by: Albert Pinkovitch Frank Rollmer
[infobox] Starring: Jean Gabin Dita Parlo Pierre Fresnay Erich von Stroheim Julien Carette Georges Péclet Marcel Dalio
[infobox] Cinematography: Christian Matras
[infobox] Edited by: Marguerite Renoir Marthe Huguet
[infobox] Music by: Joseph Kosma
[infobox] Production company: Réalisation d'Art Cinématographique
[infobox] Distributed by: Réalisation d'Art Cinématographique
[infobox] Release date: 8 June 1937 ( 1937-06-08 )
[infobox] Running time: 114 minutes
[infobox] Country: France
[infobox] Languages: French German English
[infobox] Box office: $414,620 (US re-release)
La Grande Illusion (commonly known in English as "The Grand Illusion") is a 1937 French war drama film directed by Jean Renoir , who co-wrote the screenplay with Charles Spaak . The story concerns class relationships among a small group of French officers who are German prisoners of

---
## 4. Data quality

**Overall quality.** The TMDB text is clean, and most of the problems came from joining the sources. No overview is empty, contains HTML or entities, or was detected as non-English. Five are very short (the Pan films), and the only repeated overview is the one shared by the five Small Axe films, because TMDB lists them as one series.

**What I fixed at download.** The Criterion list had spelling errors in titles and in one director's name. One film (Monsieur Hulot's Holiday) was listed twice under two Wikipedia links to the same page, so I merged the rows. Three films had picked up the TMDB record of a related film, and those, like the films automatic matching could not find, now have TMDB IDs I checked by hand.

**What remains.** No rows are dropped: `clean_catalog` records the problems in `quality_flags` on 214 films. The largest group is the 165 films with no Wikipedia article, which can only be answered from their list and TMDB fields. For 57 films the year or director disagrees with TMDB, so I leave their TMDB fields blank rather than risk corrupting the genre and runtime filters, even though some of these look like errors in the list (Diva is listed as 1961). Contempt's article is excluded because neither its year nor its director appears on the page. The other 8 articles where the year was not found are kept, since the year alone is weak evidence. A few fields are also missing for matched films (genres for 37, runtime for 9, director for 10), and those films cannot satisfy a filter on the missing field.

In [10]:
flags = audit_text_column(films, "overview")

In [11]:
findings = pd.concat([catalog_issues(films, manifest),
                      findings_table(flags, films["title"], "TMDB overview")])
findings[findings["n"] > 0]            # what you found, with examples

,where,issue,n,examples,proposed action
2,catalog,missing director,10,"Sanshiro Sugata; The Most Beautiful; Sanshiro Sugata, Part Two",Keep; fill from TMDB directors where available.
3,catalog,no Wikipedia article link,165,Martha Graham: Dance on Film; Louie Bluie; Welcome II the Terrordome,Keep for the catalog; answer from TMDB fields only and say there is no article.
5,catalog,low-confidence TMDB match,57,F for Fake; Diva; Supercop,Review tmdb_review.csv; fix by hand in id_map.json or exclude the TMDB fields for these.
6,catalog,same TMDB film on 2 rows,5,"Mangrove; Lovers Rock; Red, White and Blue",Check whether it is a wrong match or a genuine re-release; keep one row per film for retrieval.
7,catalog,matched but no genres,37,The Wold-Shadow; Kindering; I... Dreaming,Keep; exclude from filters on this field.
8,catalog,matched but no runtime,9,Tanner '88; Martha Graham: Dance on Film; The Underground Railroad,Keep; exclude from filters on this field.
13,catalog,two rows share one Wikipedia page,9,The Human Condition (film series); The Human Condition (film series); The Human Condition (film series),Redirects to the same page: merge the rows.
14,catalog,article may be the wrong film (year not found),9,Fishing with John; Contempt; The Fiances,Check by hand before using its text.
1,TMDB overview,short,5,Pan 0; Pan 2; Pan 3,"Keep, but do not embed on its own: embed it together with title, genres and keywords."
3,TMDB overview,duplicate,5,"Mangrove; Lovers Rock; Red, White and Blue",Inspect: identical texts often mean a placeholder or a wrong TMDB match. Fix the match or flag it.


In [12]:
films_clean = clean_catalog(films)     # what you do about it
films_clean

,title,year,director,film_spine,in_main_table,laserdisc_only,wiki_title,sets,revid,tmdb_id,...,tagline,overview,keywords,tmdb_directors,top_cast,poster_url,tmdb_url,match_confidence,overview_language,quality_flags
0,Grand Illusion,1937,Jean Renoir,1.0,True,False,La Grande Illusion,Essential Art House: 50 Years of Janus Films,1383674996,777,...,A Great Drama of Human Emotions,"A group of French soldiers, including the patrician Captain de Boeldieu and the working-class Lieutenant Maréchal, grapple with their own class differences after being captured and held in a World...","[prisoner, france, countryside, escape, germany, world war i, prisoner of war, brotherhood, camp, aristocrat, fraternity, war injury, escaped prisoner, prison camp, trying to escape, escape plan, ...",[Jean Renoir],"[Jean Gabin, Dita Parlo, Pierre Fresnay, Erich von Stroheim, Julien Carette, Georges Péclet, Werner Florian, Jean Dasté]",https://image.tmdb.org/t/p/w780/lWg41zE0FVixkNsFgxnlRyvDYv9.jpg,https://www.themoviedb.org/movie/777,high,en,
1,Seven Samurai,1954,Akira Kurosawa,2.0,True,False,Seven Samurai,AK 100: 25 Films by Akira Kurosawa | Essential Art House: 50 Years of Janus Films,1383674996,346,...,Unmatched for suspense and spectacle!,"A samurai answers a village's request for protection after he falls on hard times. The town needs protection from bandits, so the samurai gathers six others to help him teach the people how to def...","[martial arts, japan, samurai, sword, peasant, village, looting, rice, fencing, moral ambiguity, forbidden love, black and white, battle, period drama, bandit, practice, farmer, jidaigeki, 16th ce...",[Akira Kurosawa],"[Toshirō Mifune, Takashi Shimura, Yoshio Inaba, Seiji Miyaguchi, Minoru Chiaki, Daisuke Katō, Isao Kimura, Keiko Tsushima]",https://image.tmdb.org/t/p/w780/lOMGc8bnSwQhS4XyE1S99uH8NXf.jpg,https://www.themoviedb.org/movie/346,high,en,
2,King Kong,1933,Merian Cooper,2.0,True,True,King Kong (1933 film),NaN,1383674996,244,...,Beauty conquered the beast!,"Adventurous filmmaker Carl Denham sets out to produce a motion picture unlike anything the world has seen before. Alongside his leading lady Ann Darrow and his first mate Jack Driscoll, they arriv...","[new york city, ship, exotic island, island, monster, unsociability, screenplay, movie business, great depression, dinosaur, black and white, pre-code, sea voyage, damsel in distress, empire state...","[Merian C. Cooper, Ernest B. Schoedsack]","[Robert Armstrong, Fay Wray, Bruce Cabot, Frank Reicher, Victor Wong, James Flavin, Sam Hardy, Noble Johnson]",https://image.tmdb.org/t/p/w780/lHlnxKL5GbgRibyRFI7n1Ey850i.jpg,https://www.themoviedb.org/movie/244,high,en,
3,The Lady Vanishes,1938,Alfred Hitchcock,3.0,True,False,The Lady Vanishes,Essential Art House: 50 Years of Janus Films,1383674996,940,...,Comedy! Chills! Chuckles! in a Mystery Express!,"On a train headed for England a group of travelers is delayed by an avalanche. Holed up in a hotel in a fictional European country, young Iris befriends elderly Miss Froy. When the train resumes, ...","[espionage, secret agent, flowerpot, concussion, search for witnesses, conspiracy, black and white, train, british spy, old lady, missing person, magician, bold]",[Alfred Hitchcock],"[Margaret Lockwood, Michael Redgrave, Paul Lukas, May Whitty, Basil Radford, Naunton Wayne, Cecil Parker, Linden Travers]",https://image.tmdb.org/t/p/w780/c1t9LB76LvEARPanfEzXmkm7fwY.jpg,https://www.themoviedb.org/movie/940,high,en,
4,Amarcord,1973,Federico Fellini,4.0,True,False,Amarcord,Essential Fellini,1383674996,7857,...,The Fantastic World of Fellini!,"In an Italian seaside town, young Titta gets into trouble with his friends and watches various local eccentrics as they engage in often absurd behavior. Frequently clashing with his stern father a...","[small town, fascism, surrealism, sheik, coming of age, rural area, bathtub, semi autobiographical, 1930s]",[Federico Fellini],"[Pupella Maggio, Armando Brancia, Magali Noël, Ciccio Ingras

In [13]:
overview(films_clean, "criterion_films_enriched").sort_values(by='missing_%', ascending=False)

criterion_films_enriched: 1,928 rows x 26 columns


,dtype,non_null,missing_%,n_unique,example
sets,str,838,56.5,167,Essential Art House: 50 Years of Janus …
tagline,str,1107,42.6,1103,A Great Drama of Human Emotions
film_spine,float64,1655,14.2,1306,1.0
keywords,object,1726,10.5,1675,"['prisoner', 'france', 'countryside', '…"
wiki_title,str,1763,8.6,1760,La Grande Illusion
top_cast,object,1813,6.0,1806,"['Jean Gabin', 'Dita Parlo', 'Pierre Fr…"
genres,object,1891,1.9,417,"['Drama', 'History', 'War']"
countries,object,1913,0.8,232,['France']
languages,object,1914,0.7,341,"['Russian', 'German', 'French', 'Englis…"
runtime,Int64,1919,0.5,203,114


---
## 5. Examining the texts

**Lengths.** The texts come in very different sizes. Across the 1,763 films with an article, the article body has a median of 1,267 words (317 at the 10th percentile and 3,418 at the 90th), while the TMDB overview has a median of only 39. The plot section has a median of 484 words, but 156 articles (8.8%) have none, so for those films the system has to rely on the lead, reception and TMDB. Article length shows no trend over time: leaving aside the single film from the 1910s, the median by decade ranges from 946 words (1930s) to 1,612 (1940s).

**Frequent and distinctive words.** The most frequent plot words are generic storytelling words such as *tells*, *home*, *later* and *father*, and each of the top 30 appears in at least 22% of plots, so they say little about any one film. Genres do have their own vocabulary (*police* and *gang* for Crime, *soldiers* and *army* for War, *footage* and *interviews* for Documentary), although many distinctive words are character names or come from long series such as Godzilla and Zatoichi. Decades differ almost only by character names, which means the language does not reveal a film's period, so decade has to be a metadata filter rather than something retrieval finds.

**Hard cases.**

- **Metadata and text can disagree on the same request.** For "a tense 1970s political thriller", 36 films from the decade either carry TMDB's Thriller label or read as political and tense in their text. Only 5 do both, 28 have just the label and 3 just the text. The Confession is plainly political but counts as label only because its text has one tension word, so a filter or a word search alone would miss good answers.
- **TMDB's genres are incomplete.** In 146 films, the first sentence of the Wikipedia article names a genre TMDB lacks or shares no genre with it. Singin' in the Rain is a musical with no Music tag, and The Silence of the Lambs is called a horror film with no Horror tag, so a hard genre filter will drop some correct films. The comparison is only a keyword match and has false alarms: Alphaville is labeled Adventure because its subtitle contains the word "aventure".
- **Some articles are too thin to answer from.** The shortest have 17 to 49 words and usually no plot section (The Chorus, Humain, trop humain), so those films depend on their TMDB text. Four of the ten shortest are flagged as not English, but all four are English sentences containing a title in Persian or Japanese script, so that flag is wrong for them and I did not exclude them.

In [14]:
corpus = build_corpus(films, manifest)                 

In [15]:
length_summary(corpus); length_summary(corpus, "decade")        

,n,median,p10,p90
decade,,,,
1910,1,452.0,452.0,452.0
1920,30,1374.0,441.0,2451.0
1930,139,946.0,272.0,2992.0
1940,132,1612.0,450.0,3537.0
1950,210,1246.0,434.0,3223.0
1960,363,1164.0,283.0,3395.0
1970,275,1276.0,255.0,3804.0
1980,221,1212.0,307.0,3441.0
1990,200,1396.0,479.0,3131.0


In [16]:
word_frequencies(corpus["plot"], top=30)

,word,count,docs,doc_share
0,tells,1404,700,0.40
1,man,1350,679,0.39
2,home,1329,722,0.41
3,later,1207,740,0.42
4,back,1169,714,0.40
5,father,1097,497,0.28
6,mother,1044,468,0.27
7,night,1031,615,0.35
8,police,956,415,0.24
9,time,940,624,0.35


In [17]:
distinctive_words(corpus, "genres", "plot", multi=True) # also "decade", "country"

,group,n_docs,distinctive words
0,Action,124,"godzilla, ichi, wong, fight, bond, samurai, sword, clan"
1,Adventure,120,"ichi, bond, sword, lawrence, clan, gold, valerie, duke"
2,Animation,12,"cat, dogs, tetsuo, wall, kobayashi, farmers, saki, warren"
3,Comedy,381,"louis, sam, harold, christine, charles, fanny, annie, jordan"
4,Crime,252,"police, money, gang, murder, prison, gun, tony, shoots"
5,Documentary,61,"footage, interviews, gates, documentary, band, bowie, strike, dylan"
6,Drama,1213,"mother, relationship, young, son, anna, family, asks, day"
7,Family,20,"dorothy, jason, balloon, cat, wall, wee, alec, kevin"
8,Fantasy,94,"baron, princess, kong, belle, adam, dorothy, valerie, daniel"
9,History,104,"catherine, king, army, lawrence, battle, martin, tale, merrick"


In [18]:
# Hard case 1
partial_matches(corpus)[:10]                                  

,category,title,year,tmdb_genres,political_hits,tension_hits,first_sentence
633,both,Investigation of a Citizen Above Suspicion,1970,"[Drama, Thriller]","[corruption, ministry]","[investigation, thriller]","Investigation of a Citizen Above Suspicion ( Italian: Indagine su un cittadino al di sopra di ogni sospetto ) is a 1970 Italian satirical crime thriller film directed by Elio Petri, starring Gian ..."
967,both,Sweet Sweetback's Baadasssss Song,1971,"[Action, Crime, Drama, Thriller]","[president, revolutionary]","[fear, murdered, thriller]","Sweet Sweetback's Baadasssss Song is a 1971 American independent blaxploitation action thriller film written, co-produced, scored, edited, directed by, and starring Melvin Van Peebles."
689,both,State of Siege,1972,"[Thriller, Drama]","[government, political, terrorism]","[killed, thriller]",State of Siege ( French: État de siège ) is a 1972 French–Italian–West German political thriller film directed by Costa-Gavras starring Yves Montand and Renato Salvatori.
1372,both,Lone Wolf and Cub: Sword of Vengeance,1972,"[Drama, Action, Adventure, Thriller]","[assassinate, conspiracy]","[assassin, killed, spies]","Lone Wolf and Cub: Sword of Vengeance ( Japanese: 子連れ狼 子を貸し腕貸しつかまつる, Hepburn: Kozure Ōkami: Kowokashi udekashi tsukamatsuru ) is a 1972 Japanese chambara film directed by Kenji Misumi."
939,both,The Parallax View,1974,"[Crime, Drama, Thriller]","[assassination, political, president, senator]","[assassin, investigates, investigation, killed, murder, paranoia, paranoid, thriller]","The Parallax View is a 1974 American neo-noir political thriller film starring Warren Beatty, Paula Prentiss, Hume Cronyn, William Daniels, Kenneth Mars, Walter McGinn, Kelly Thordsen and Jim Davi..."
234,metadata only,Le Cercle Rouge,1970,"[Crime, Drama, Thriller]",[],[tense],"Le Cercle Rouge ( French pronunciation: [lə sɛʁkl ʁuʒ]; ""The Red Circle"") is a 1970 French crime film written and directed by Jean-Pierre Melville."
688,metadata only,The Confession,1970,"[Drama, Thriller]","[communist, government, minister]",[surveillance],The Confession ( French: L'aveu ) is a 1970 French-Italian film directed by Costa-Gavras starring Yves Montand and Simone Signoret.
203,metadata only,Straw Dogs,1971,"[Thriller, Drama, Crime]",[],[thriller],Straw Dogs is a 1971 psychological thriller film directed by Sam Peckinpah and starring Dustin Hoffman and Susan George.
878,metadata only,Klute,1971,"[Thriller, Crime, Drama]",[president],"[investigation, killed, murder, murdered, paranoia, paranoid, surveillance, thriller]",Klute is a 1971 American neo-noir psychological thriller film directed and produced by Alan J.
1001,metadata only,Shaft,1971,"[Crime, Thriller, Drama, Action]",[],"[investigator, kidnapped, killed, thriller]",Shaft is a 1971 American blaxploitation crime action thriller film directed by Gordon Parks and written by Ernest Tidyman and John D.


In [19]:
# Hard case 2
genre_text_disagreement(corpus)

,title,year,kind,wikipedia_says,tmdb_says,first_sentence
0,The Silence of the Lambs,1991,text names a genre TMDB lacks,"Horror, Thriller","Crime, Drama, Thriller","The Silence of the Lambs is a 1991 American psychological horror thriller film directed by Jonathan Demme and written by Ted Tally, adapted from Thomas Harris 's 1988 novel."
1,Young and Innocent,1937,text names a genre TMDB lacks,"Crime, Thriller","Mystery, Romance, Thriller","Young and Innocent, released in the US as The Girl Was Young, is a 1937 British crime thriller film directed by Alfred Hitchcock and starring Nova Pilbeam and Derrick De Marney."
2,Alphaville,1965,no overlap,Adventure,"Drama, Mystery, Science Fiction","Alphaville (subtitled Une étrange aventure de Lemmy Caution, ""A Strange Adventure of Lemmy Caution"" ) is a 1965 French New Wave tech noir film written and directed by Jean-Luc Godard, and starring..."
3,M,1931,text names a genre TMDB lacks,"Mystery, Thriller","Crime, Drama, Thriller","M is a 1931 German mystery thriller film directed by Fritz Lang and starring Peter Lorre as Hans Beckert, a serial killer who targets children, in his third screen role."
4,Adam's Rib,1949,text names a genre TMDB lacks,"Comedy, Drama, Romance","Comedy, Romance",Adam's Rib is a 1949 American romantic comedy-drama film by director George Cukor from a screenplay written by Ruth Gordon and Garson Kanin.
...,...,...,...,...,...,...
141,The League of Gentlemen,1960,text names a genre TMDB lacks,"Action, Comedy","Comedy, Crime","The League of Gentlemen is a 1960 British heist action comedy film directed by Basil Dearden and starring Jack Hawkins, Nigel Patrick, Roger Livesey and Richard Attenborough."
142,Jungle Book,1942,text names a genre TMDB lacks,"Action, Adventure","Adventure, Fantasy",Jungle Book or Rudyard Kipling's Jungle Book is a 1942 American action-adventure film in Technicolor independently produced by the Korda brothers and loosely adapted from Rudyard Kipling 's 1894 s...
143,Genocide,1968,text names a genre TMDB lacks,"Horror, Science Fiction, War","Horror, Science Fiction","Genocide ( Japanese: 昆虫大戦争, Hepburn: Konchu daisenso; lit. ' Great Insect War ' ), also known as War of the Insects, is a 1968 Japanese science fiction horror film directed by Kazui Nihonmatsu."
144,Douce,1943,text names a genre TMDB lacks,"Drama, Romance",Drama,"Love Story (French: Douce ) is a 1943 French historical romantic drama film directed by Claude Autant-Lara and starring Odette Joyeux, Madeleine Robinson and Marguerite Moreno."


In [20]:
# Hard case 3
thin_texts(corpus)

,problem,title,year,n_words_body,n_words_plot,lead_language,lead
1750,not English,The Chorus,1982,17,0,ar,"The Chorus ( Persian: همسرایان, Hamsorayan ) is a 1982 Iranian short film directed by Abbas Kiarostami."
1604,no plot section,"Humain, trop humain",1973,20,0,en,"Humain, trop humain is a French documentary film by Louis Malle about the operations of a Citroën car production plant."
468,no plot section,Bergman Island,2006,26,0,en,"Bergman Island is a 2004 documentary film directed by Marie Nyreröd and starring Ingmar Bergman. It screened at various film festivals, including the Gothenburg Film Festival."
1752,not English,First Graders,1984,33,0,ar,"First Graders ( Persian:...اولیها, Avvalihā... ) is a 1984 Iranian documentary feature film directed by Abbas Kiarostami, in which a hidden camera follows a group of first graders during a day at ..."
1198,no plot section,Carl Th. Dreyer - My Metier,1995,35,0,en,Carl Th. Dreyer: My Metier ( Danish: Carl Th. Dreyer: Min metier ) is a 1995 Danish documentary film directed by Torben Skjødt Jensen [ da ] about the film career of Carl Theodor Dreyer.
1728,no plot section,David Golder,1930,38,0,en,"David Golder is a 1931 French drama film directed by Julien Duvivier and starring Harry Baur, Paule Andral and Jackie Monnier. It is an adaptation of Irène Némirovsky 's 1929 novel David Golder, a..."
1663,not English,Japanese Summer: Double Suicide,1967,45,22,zh,"Double Suicide: Japanese Summer ( Japanese: 無理心中日本の夏, Hepburn: Muri shinju: Nihon no natsu ) is a 1967 Japanese film directed by Nagisa Ōshima."
1736,no plot section,The Count of the Old Town,1935,45,0,en,"The Count of the Old Town ( Swedish: Munkbrogreven ) is a 1935 Swedish comedy film directed by Edvin Adolphson and Sigurd Wallén, both of whom also had major acting roles in the film. It was Ingri..."
1606,no plot section,Calcutta,1969,47,0,en,"Calcutta is a 1969 French documentary film about Calcutta, directed by Louis Malle. Most of the footage was shot in Calcutta in 1967-1968 alongside Mrinal Sen who was then shooting footage for his..."
1751,not English,Fellow Citizen,1983,49,31,ar,"Fellow Citizen ( Persian: همشهری, romanized: Hamshahri ) is a 1983 Iranian documentary film directed by Abbas Kiarostami."


---
## 6. The documents and the evaluation set

### Preparing the documents

**Which texts the system will search.** The Wikipedia articles (long-form text) and the TMDB overviews (short descriptions). The system searches both, plus a fact card for each film. The current build has 27,576 chunks saved in `data/processed/chunks.jsonl`:

- **Catalog facts (1,928 chunks):** a short card built from the Criterion list and TMDB, with title, year, director, country, genres, runtime, cast and spine number. It covers the "which films..." questions.
- **TMDB overviews (1,871 chunks):** overview, tagline and keywords.
- **Wikipedia articles (23,777 chunks):** 1,757 film articles, split into the infobox, the lead and each article section, labeled by type (plot, cast, production, release, reception, awards, themes and legacy, other). This group also includes 21 chunks from the page about The Criterion Collection itself, which answers questions about the company. That page is not a film, so its film fields are empty and it can never satisfy a film filter.

Films with no usable article (168) can still be found through their catalog and TMDB chunks. Che: Part One and Part Two share one article, as do the three parts of The Human Condition and the two parts of Ivan the Terrible and Fårö Document, so each film keeps its own catalog and TMDB chunks and the article is chunked only once.

**How I chunk them.** I follow the structure of the article instead of cutting at a fixed length. A chunk never crosses a section heading. Paragraphs are merged up to about 220 words, with a soft limit of 320, and one sentence is repeated at the start of the next chunk in the same section. Tails under 40 words are merged into the previous chunk, and sections under 15 words are dropped. Catalog facts, TMDB text and infoboxes stay as one chunk each. The result has a median of 130 words (10th percentile 45, 90th percentile 209). Each chunk also stores an `embed_text` version that starts with a line naming the film, year and section, so that it still makes sense on its own, while `text` stays clean for display.

**What metadata each chunk keeps.** My data has no ratings, and the closest equivalents to "product" and "date" are the film and its year and article revision.

- **Identity and citation:** `chunk_id`, `film_id`, `title`, `source`, `source_title`, `section`, `chunk_index`, `n_chunks`, `revid`, `license` and `url`. For Wikipedia, `url` is a permalink to the exact revision and section, so a citation always points to the text the system used.
- **Filtering:** `year`, `decade`, `director`, `country`, `genres`, `runtime`, `film_spine` and `in_main_table`, which support the hard constraints in the evaluation.
- **Behavior and display:** `section_type`, `spoilers` (true for plot chunks, so a recommendation can avoid revealing endings), `quality_flags`, `tmdb_id`, `poster_url`, `tmdb_url` and `n_words`.

**What I correct and leave out.** Eight spelling errors in the source list, such as "The Wages for Fear" and "Harikari", are corrected when the data is loaded, so the chunks carry the right titles. Two more title typos and one misspelled director are corrected when the list is downloaded. Two box sets, the Beastie Boys Video Anthology and 100 Years of Olympic Films, are left out of the catalog. A film with no spine number of its own takes its box set's spine, so several films can share one number. One article (Contempt) is excluded because neither the film's year nor its director appears on the page, so it is probably the wrong article. TMDB fields are left blank for low-confidence matches, because a wrong match would corrupt the genre and runtime filters.

In [21]:
chunks, excluded = build_chunks(films, manifest)   # or: python prepare_documents.py
save_chunks(chunks)    

### The evaluation set

The evaluation set is saved in `data/evaluation_set.csv`: 30 questions a film fan or cineclub organizer would really ask, written before building the system. 25 can be answered from the data and 5 cannot. Each row records the question, where the evidence should come from (`L` = Criterion list, `T` = TMDB, `A` = Wikipedia article), the expected answer, the expected sources when specific ones hold the answer, and an automatic check.

- **Recommendations and programming (1–6):** these are matters of taste with no single right answer, so the expected answer is defined by what can be checked. Picks must be in the catalog and meet the decade, genre, country or runtime constraints, the similarity question must not recommend its seed film (5), and the double feature must span two decades (6). Tone and fit are not scored. Each of these six questions also has a shortlist of catalog films, which gives the expected sources for the source hit rate. I wrote the shortlists after the first system runs, so they do not affect whether a question passes its automatic check.
- **Catalog sets and facts (7–11):** Brazilian productions or films shot in Brazil (8 required films), Wim Wenders films (10), the films added in November 2026 (three new titles, stated only in their articles), the spine number of Seven Samurai (2), and whether Citizen Kane was the first Criterion release (the first releases were LaserDiscs, while the first DVD was Grand Illusion). The Brazil and Wim Wenders sets are computed from the catalog, and the November 2026 question is checked against its three titles. Question 7 deliberately needs two kinds of evidence: TMDB production countries, and the article stating that Notorious shot footage in Rio de Janeiro.
- **Film details (12, 13):** what Burden of Dreams is about and the runtime of Seven Samurai (207 minutes in the stored data). Each one names the source that answers it.
- **Harder reasoning (14–15):** how many 1970s films are Italian productions (34 in the catalog), which I expect plain retrieval to struggle with, and a two-step question that goes from spine 11 (The Seventh Seal) to Ingmar Bergman's 8 films from the 1970s.
- **Mood recommendations (16–20):** surreal and dreamlike, melancholy love stories, gentle and hopeful films from Iran, whimsical French comedies, and tense thrillers. Hard filters are checked automatically and mood itself is not scored. TMDB keywords give a separate cross-check, because the tags are too sparse and noisy to pass or fail a film on.
- **Plot identification (21–23):** a plot description with no title, for three lesser-known films: Adoption, The Warped Ones and Journey to the Beginning of Time. The check is whether the system names the right film.
- **Plot keyword lists (24–25):** films where a fisherman matters to the story, and films where hypnosis plays a part. An answer passes when it names at least two films with the matching TMDB keyword.
- **Questions the data cannot answer (26–30):** who directed The Conformist and which novel it adapts (a well-known film that is not in the Collection, and no stored article names the novel), which films are streaming in Germany this week (live data), how much Criterion paid to license Seven Samurai (undocumented), which films will join the Collection in December 2027 (the future), and which Collection film is about a weatherman reliving the same day (Groundhog Day is not in the Collection). The correct behavior is to refuse, and an answer that does not refuse fails its check. Questions 26 and 27 are the most direct tests of the costliest mistake: answering from general knowledge about something outside the Collection.

When scoring, a film counts under its catalog title or any accepted alternate title from TMDB or Wikipedia, so that "Se7en" matches "Seven".

**Status of the set:** every question now has its expected answer, including the shortlists for questions 1–6. I added questions 9 and 30 after the first runs.

In [22]:
eval_set = pd.read_csv(ROOT / "data" / "evaluation_set.csv")
print(f"{len(eval_set)} questions: {(eval_set['answerable'] == 1).sum()} marked answerable, "
      f"{(eval_set['answerable'] == 0).sum()} the data cannot answer")
eval_set[["id", "group", "question", "evidence", "expected_answer", "expected_sources", "auto_check"]]

30 questions: 25 marked answerable, 5 the data cannot answer


,id,group,question,evidence,expected_answer,expected_sources,auto_check
0,1,recommendation,"What should I watch if I want a tense, 1970s political thriller?",L+T+A,Shortlist of 1970s Thriller films with a political plot: The Parallax View; State of Siege; The Confession; Investigation of a Citizen Above Suspicion,A:The Parallax View|A:State of Siege|A:The Confession|A:Investigation of a Citizen Above Suspicion,constraints
1,2,recommendation,"I want a slow, atmospheric science-fiction film from the 1980s or 1990s.",L+T+A,Shortlist of 1980-99 Science Fiction films with a slow or atmospheric feel: Blade Runner; Until the End of the World; 1984; Testament; The Element of Crime,A:Blade Runner|A:Until the End of the World|A:1984|A:Testament|A:The Element of Crime,constraints
2,3,recommendation,Which Japanese films from the 1960s deal with war?,L+T+A,Shortlist of 1960s Japanese films that deal with war or its aftermath: The Human Condition III: A Soldier's Prayer; Story of a Prostitute; Onibaba; Gate of Flesh; Pigs and Battleships,T:The Human Condition III: A Soldier's Prayer|A:Story of a Prostitute|A:Onibaba|A:Gate of Flesh|A:Pigs and Battleships,constraints
3,4,recommendation,"Suggest a light comedy for a cineclub night, under 100 minutes.",L+T,Shortlist of light comedies of 99 minutes or less: Monsieur Hulot's Holiday; Trouble in Paradise; The Lady Eve; His Girl Friday; The Palm Beach Story; The Awful Truth; Safety Last!; A Hard Day's N...,A:Monsieur Hulot's Holiday|A:Trouble in Paradise|A:The Lady Eve|A:His Girl Friday|A:The Palm Beach Story|A:The Awful Truth|A:Safety Last!|A:A Hard Day's Night,constraints
4,5,similarity,I loved Seven Samurai. What else should I watch?,A+T,"Shortlist sharing director, era or genre with Seven Samurai: Yojimbo; Sanjuro; The Hidden Fortress; Throne of Blood; Harakiri; Samurai I: Musashi Miyamoto; Three Outlaw Samurai",A:Seven Samurai|A:Yojimbo|A:Sanjuro|A:The Hidden Fortress|A:Throne of Blood|A:Harakiri|A:Samurai I: Musashi Miyamoto|A:Three Outlaw Samurai,seed_excluded
5,6,programming,"Plan a double feature on paranoia and surveillance, with the two films from different decades.",L+A,Candidate pairs from different decades whose plots involve paranoia or surveillance: The Parallax View + Blow Out; Klute + Three Colors: Red; Peeping Tom + Lost Highway,A:The Parallax View|A:Blow Out|A:Klute|A:Three Colors: Red|A:Peeping Tom|A:Lost Highway,decades_differ
6,7,catalog set,Which Brazilian productions or films shot in Brazil are in the Collection?,L+T+A,"Set = rows with TMDB production country Brazil (Black Orpheus; Black God, White Devil; Kiss of the Spider Woman; The Secret Agent; Limite; Pixote; São Paulo, Incorporated) plus films shot in Brazi...","T:Black Orpheus|T:Black God, White Devil|T:Kiss of the Spider Woman|T:The Secret Agent|T:Limite|T:Pixote|T:São Paulo, Incorporated|A:Notorious",set_from_catalog
7,8,catalog set,Which Wim Wenders films are in the Collection?,L,"Set = all L rows with director Wim Wenders (10 films: Alice in the Cities #814, Wrong Move #815, Kings of the Road #816, The American Friend #793, Paris, Texas #501, Wings of Desire #490, Until th...",L:Wim Wenders,set_from_catalog
8,9,catalog set,Which films were added to the Criterion Collection in November 2026?,A,"KPop Demon Hunters (November 3), Marie Antoinette (November 10) and The Right Stuff (November 24), each with a new spine number. Harold and Maude's 4K edition on November 3 is a reissue of spine 6...",A:KPop Demon Hunters|A:Marie Antoinette|A:The Right Stuff,contains_groups
9,10,catalog fact,What is the spine number of Seven Samurai?,L,2,L:Seven Samurai,number


---
## 7. Baseline report: the LLM alone

**What I ran.** I sent each of the 30 evaluation questions once to `gpt-6-luna`, with no retrieval and no tools. The instructions told it to recommend only Criterion films, refuse when it could not answer reliably, and return its answer, picks and refusal as JSON. The answers go through the same `score_run` checks my final system will face. The run is saved in `data/processed/baseline_llm_alone.json`, so these numbers can be reproduced without new API calls.

| Metric | Result |
|---|---|
| Automatic checks passed (answerable questions) | 11 of 25 (44.0%) |
| Automatic precision@k on the pick questions (1–6, 16–20) | 0.70 |
| Source hit rate | 0% |
| False refusals (answerable questions) | 3 of 25 (12.0%) |
| Refusals on unanswerable questions | 4 of 5 (80%) |
| Tokens | 6,923 input, 18,124 output |
| Cost | $0.0098 in total, $0.00033 per question |
| Time | 8.9 seconds per question on average, 268 seconds in total |

Precision counts the picks that are in the catalog and meet the hard constraints. It does not measure tone or fit. The source hit rate is 0% by construction, since a model without retrieval has no stored source to cite. The cost uses a price of $0.10 per million input tokens and $0.50 per million output tokens. These figures come from a single run, and the model's answers vary.

**Results by question type.**

| Type | Automatic checks passed |
|---|---|
| Recommendations, similarity and programming (1–6) | 3 of 6 |
| Catalog sets and facts (7–11) | 1 of 5 |
| Film details (12, 13) | 2 of 2 |
| Aggregation and multi-step (14–15) | 0 of 2 |
| Mood recommendations (16–20) | 3 of 5 |
| Plot identification (21–23) | 2 of 3 |
| Plot keyword lists (24–25) | 0 of 2 |

**Keyword support for the mood questions.** This is a cross-check and does not affect the pass rate. It counts how many of the three picks carry a TMDB keyword for the requested mood. The base rate is the share of eligible films with such a tag, which is roughly what random picks would score.

| Question | Picks tagged | Tagged among eligible films | Base rate |
|---|---|---|---|
| 16. Surreal and dreamlike, 1990 or later | 2 of 3 | 13 of 381 | 3.4% |
| 17. Melancholy love story | 1 of 3 | 10 of 366 | 2.7% |
| 18. Gentle and hopeful, from Iran | 1 of 3 | 4 of 18 | 22.2% |
| 19. Playful, whimsical French comedy | 2 of 3 | 4 of 82 | 4.9% |
| 20. Tense thriller, 1980 or later | 1 of 3 | 12 of 80 | 15.0% |

Overall, 7 of the 15 possible picks are tagged (46.7%), against an average base rate of 9.6%. The model's mood choices line up with the tags far more often than chance would, although this is a small sample. Question 18 counts as 1 of 3 because the model named only one film. A missing tag does not make a pick wrong, since In the Mood for Love and Brief Encounter are untagged but plausible answers to question 17, which is why this stays a cross-check.

**Where it failed.**

- **Films outside my catalog:** 8 of the 41 titles it named are not in it: The Sacrifice (2), The Conversation and Caché (6), The Little Theatre of Jean Renoir (15), La terra trema and Leviathan (24), and The Cabinet of Dr. Caligari and Dr. Mabuse, the Gambler (25). This is the costliest kind of mistake in my framing.
- **Plot keyword lists:** both failed because every film it named was either outside the catalog or had no matching tag. This is where it is weakest against my catalog.
- **Incomplete or wrong lists:** it found 1 of the 8 required films for question 7 and 6 of 10 Wim Wenders films (8). On question 15 it named the wrong director for spine 11.
- **Too few picks:** it named only one film for questions 3 and 18, where three were needed.
- **Answering outside the Collection:** it answered question 26 about The Conformist from general knowledge instead of refusing.
- **False refusals:** it refused the question about films added in November 2026 (9) and the counting question (14), and could not identify The Warped Ones from its plot (22).
- **Other failures:** on question 11 it said Citizen Kane was the first release without separating LaserDisc from DVD.

**What this batch showed.** The model identified two of the three lesser-known films from their plots and admitted it did not know the third instead of inventing a title. So plot descriptions alone were only moderately hard for it. It struggles most when an answer depends on what is actually in my catalog, as in the keyword lists, the set questions and the counting question.

In [23]:
cat = prepare_catalogue(films_clean)
answers, baseline_log = run_baseline(eval_set)        # saved run; refresh=True calls the model again

baseline_results = score_run(eval_set, answers, cat, "llm_alone")
baseline_results["pick_precision"] = [pick_precision(q, answers[q["id"]], cat) for _, q in eval_set.iterrows()]
baseline_results = baseline_results.merge(baseline_log[["id", "input_tokens", "output_tokens", "seconds"]], on="id")
baseline_results.merge(eval_set[["id", "question"]], on="id")[
    ["id", "question", "picks", "answer", "auto_pass", "auto_detail", "refused", "pick_precision",
     "input_tokens", "output_tokens", "cost_usd", "seconds"]]

,id,question,picks,answer,auto_pass,auto_detail,refused,pick_precision,input_tokens,output_tokens,cost_usd,seconds
0,1,"What should I watch if I want a tense, 1970s political thriller?",State of Siege | The Confession | Investigation of a Citizen Above Suspicion,"For a tense 1970s political thriller, try these three:",True,all picks meet constraints,False,1.000000,234,341,0.000194,7.60
1,2,"I want a slow, atmospheric science-fiction film from the 1980s or 1990s.",Until the End of the World | Naked Lunch | The Sacrifice,"For slow, atmospheric science fiction, try these Criterion Collection films:",False,"Naked Lunch: genres ['Crime', 'Drama'] lack Science Fiction; The Sacrifice: not in catalogue",False,0.333333,237,347,0.000197,6.16
2,3,Which Japanese films from the 1960s deal with war?,The Human Condition III: A Soldier’s Prayer,"The clear match is The Human Condition III: A Soldier’s Prayer (1961), about a Japanese soldier’s experiences at the end of World War II. It’s the final part of a trilogy; the first two installmen...",False,"only 1 picks, need 3",False,0.333333,230,958,0.000502,11.08
3,4,"Suggest a light comedy for a cineclub night, under 100 minutes.",His Girl Friday | Good Morning | House,"For a light cineclub night, try one of these Criterion comedies—all under 100 minutes:",True,all picks meet constraints,False,1.000000,232,971,0.000509,11.53
4,5,I loved Seven Samurai. What else should I watch?,Yojimbo | Sanjuro | The Hidden Fortress,"If you loved the camaraderie, action, and samurai setting of Seven Samurai, try these next:",True,ok,False,1.000000,228,135,0.000090,2.83
5,6,"Plan a double feature on paranoia and surveillance, with the two films from different decades.",The Conversation | Caché,Pair a classic tale of surveillance paranoia with a modern thriller about being watched.,False,"need two picks, both in the catalogue",False,0.000000,234,99,0.000073,2.71
6,7,Which Brazilian productions or films shot in Brazil are in the Collection?,Black Orpheus,Black Orpheus is the Criterion Collection title I can reliably identify that was produced in Brazil or filmed there.,False,"recall 0.12, precision 1.00, missing ['black god white devil', 'kiss of the spider woman', 'limite', 'notorious', 'pixote', 'sao paulo incorporated', 'secret agent']",False,NaN,230,976,0.000511,12.80
7,8,Which Wim Wenders films are in the Collection?,"Alice in the Cities | The American Friend | Kings of the Road | Paris, Texas | Wings of Desire | Wrong Move",6 Wim Wenders films are in the Collection.,False,"recall 0.60, precision 1.00, missing ['buena vista social club', 'perfect days', 'pina', 'until the end of the world']",False,NaN,227,574,0.000310,6.68
8,9,Which films were added to the Criterion Collection in November 2026?,,I don't know which films will be added to the Criterion Collection in November 2026.,False,"missing one of: [['kpop demon hunters', 'k pop demon hunters'], ['marie antoinette'], ['right stuff']]",True,NaN,231,68,0.000057,10.59
9,10,What is the spine number of Seven Samurai?,,2,True,"expected 2 +/- 0, found [2.0]",False,NaN,226,53,0.000049,1.93


In [24]:
print(f"automatic precision@k, pick questions: {baseline_results['pick_precision'].mean():.2f}")
print(f"tokens: {baseline_log['input_tokens'].sum():,} input, {baseline_log['output_tokens'].sum():,} output")
print(f"cost: ${baseline_log['cost_usd'].sum():.4f} total, ${baseline_log['cost_usd'].mean():.5f} per question")
print(f"time: {baseline_log['seconds'].mean():.1f} s per question, {baseline_log['seconds'].sum():.0f} s total")
summarize(baseline_results, eval_set)

automatic precision@k, pick questions: 0.70
tokens: 6,923 input, 18,124 output
cost: $0.0098 total, $0.00033 per question
time: 8.9 s per question, 268 s total


n_answerable                 25.000000
auto_pass_rate                0.440000
retrieval_hit_rate            0.000000
false_refusal_rate            0.120000
refusal_rate_unanswerable     0.800000
mean_cost_usd                 0.000325
dtype: float64

In [25]:
keyword_table = keyword_support(eval_set, answers, cat)
print(f"picks with a matching TMDB keyword: {keyword_table['picks_tagged'].sum()} of {keyword_table['k'].sum()}, "
      f"average base rate {keyword_table['base_rate'].mean():.1%}")
keyword_table.drop(columns="question")

picks with a matching TMDB keyword: 7 of 15, average base rate 9.6%


,id,picks,tagged_picks,picks_tagged,k,share_tagged,eligible_tagged,eligible,base_rate
0,16,Mulholland Dr. | The Double Life of Véronique | The Tree of Life,Mulholland Dr. | The Tree of Life,2,3,0.67,13,381,0.034
1,17,In the Mood for Love | Brief Encounter | The Umbrellas of Cherbourg,The Umbrellas of Cherbourg,1,3,0.33,10,366,0.027
2,18,Where Is the Friend's House?,Where Is the Friend's House?,1,3,0.33,4,18,0.222
3,19,PlayTime | Mon Oncle | Monsieur Hulot's Holiday,PlayTime | Mon Oncle,2,3,0.67,4,82,0.049
4,20,Cure | Blow Out | The Vanishing,Cure,1,3,0.33,12,80,0.150


---
## 8. Evaluating the baseline

**How good is the baseline?** It is a useful starting point, but it is not a product I could trust. The LLM alone passed the automatic checks on 11 of the 25 answerable questions (44%), and on average 0.70 of the picks per recommendation question were in the catalog and met the hard constraints. It is cheap, at about $0.01 for all 30 questions, and it does well on what general film knowledge can answer: it passed 3 of the 5 mood questions, identified 2 of the 3 lesser-known films from their plots, and refused 4 of the 5 questions it should refuse. These numbers come from one run of a model whose answers vary, and they do not measure tone or fit, so I read them as a rough level and not as a precise score.

**Which mistakes does it make?** Its main weakness is that it does not know what is actually in my catalog. Of the 41 titles it named, 8 are not in the catalog, which is the costliest mistake in my framing because a user could plan a screening around a film that is not in the Collection. The same weakness explains the worst results: it failed both plot keyword lists, found only 1 of the 8 required films for the Brazil question and 6 of 10 Wim Wenders films, named the wrong director for spine 11, and declined to give a count of Italian productions. It also answered the question about The Conformist from general knowledge instead of saying the film is outside the Collection. Finally, none of its answers can be verified, because it has no stored source to cite. That leaves a source hit rate of 0%.

**What do I expect my solution to fix?** Retrieval should address the two problems that matter most. Answering only from retrieved Criterion documents should reduce the number of films from outside the Collection, make questions like The Conformist easier to refuse, and let every answer cite the Wikipedia section or TMDB record it used. Metadata filters on year, country, genre and director should also help the recommendation questions meet their hard constraints.

I do not expect plain retrieval to fix everything. Complete lists and counts need every matching film, while retrieval returns only the top few passages, so questions 7, 8, 14 and 15 will probably need metadata filters or a pandas tool. Mood is also uncertain, because the baseline already does reasonably well there and the keyword tags are sparse, so the automatic checks may not show whether retrieval really helps. There are risks too: a stricter system may refuse more answerable questions than the baseline's 3 of 25, and retrieval will add cost and time. My solution has to beat 44% on the automatic checks and 0.70 on precision, raise the source hit rate from 0%, and refuse all five unanswerable questions, without giving up the low false-refusal rate.